# 01. Preparación de Datos Históricos y Cálculo de Agregaciones (GHAW-H)

**Objetivo:** Cargar las 4 tablas históricas del dataset GHAW-H, reconstruir la secuencia de versiones y transiciones, calcular métricas del body y frontmatter (PyYAML), y generar tablas agregadas por archivo y mes.

In [ ]:
import os
import yaml
import pandas as pd
import numpy as np

# Crear directorio de datos procesados
PROCESSED_DIR = "data/processed"
RAW_DIR = "data/raw"  # Cambiar según la ruta de almacenamiento de GHAW-H
os.makedirs(PROCESSED_DIR, exist_ok=True)

## 1. Carga y Relación de las Tablas Históricas

In [ ]:
df_repo = pd.read_parquet(os.path.join(RAW_DIR, "repository.parquet"))
df_history = pd.read_parquet(os.path.join(RAW_DIR, "source_markdown_file_history.parquet"))
df_version = pd.read_parquet(os.path.join(RAW_DIR, "source_markdown_file_version.parquet"))
df_snapshot = pd.read_parquet(os.path.join(RAW_DIR, "source_markdown_file_snapshot.parquet"))

# Recuento de Entidades
n_repos = df_repo["repository_id"].nunique()
n_histories = df_history["source_markdown_file_history_id"].nunique()
n_versions = df_version["source_markdown_file_version_id"].nunique()

# Historias con 1 versión vs >= 2 versiones
versions_per_history = df_version.groupby("source_markdown_file_history_id")["source_markdown_file_version_id"].count()
single_version_histories = (versions_per_history == 1).sum()
multi_version_histories = (versions_per_history >= 2).sum()

summary_counts = pd.DataFrame({
    "Métrica": ["Repositorios Únicos", "Historias de Archivo Únicas", "Versiones Totales", 
                "Historias con 1 Versión", "Historias con >=2 Versiones"],
    "Cantidad": [n_repos, n_histories, n_versions, single_version_histories, multi_version_histories]
})
display(summary_counts)

## 2. Reconstrucción de las Secuencias de Versiones

In [ ]:
# Convertir fechas a Datetime
df_version["committed_at"] = pd.to_datetime(df_version["committed_at"], errors="coerce")

# Ordenamiento por historia y fecha/orden
df_version = df_version.sort_values(
    by=["source_markdown_file_history_id", "committed_at"]
).reset_index(drop=True)

# Asignación de número de versión ordenado dentro de la historia
df_version["version_rank"] = df_version.groupby("source_markdown_file_history_id").cumcount() + 1

# Detección de anomalías de fechas
null_dates = df_version["committed_at"].isnull().sum()
print(f"Versiones con fechas nulas/inválidas: {null_dates}")

# Mergear con predecessors para formar la tabla de transiciones
df_transitions = df_version.merge(
    df_version,
    left_on="predecessor_source_markdown_file_version_id",
    right_on="source_markdown_file_version_id",
    suffixes=("_curr", "_prev"),
    how="inner"
)

# Validar pertenencia a la misma historia
same_history = (df_transitions["source_markdown_file_history_id_curr"] == df_transitions["source_markdown_file_history_id_prev"]).all()
print(f"¿Todas las transiciones corresponden a la misma historia?: {same_history}")

## 3. Cálculo de Medidas (Body, Frontmatter con PyYAML y Tiempos)

In [ ]:
def extract_yaml_and_body_metrics(content_text):
    if not isinstance(content_text, str) or not content_text.strip():
        return 0, 0, True, False  # word_count, yaml_keys, is_empty, is_yaml_error

    # Separar por espacios para contar palabras del body
    words = len(content_text.split())
    
    yaml_keys = np.nan
    is_yaml_error = False
    
    if content_text.startswith("---"):
        parts = content_text.split("---", 2)
        if len(parts) >= 3:
            try:
                parsed = yaml.safe_load(parts[1])
                if isinstance(parsed, dict):
                    # Solo claves de primer nivel
                    yaml_keys = len(parsed.keys())
                else:
                    yaml_keys = 0
            except Exception:
                is_yaml_error = True
                yaml_keys = np.nan
        else:
            yaml_keys = 0
    else:
        yaml_keys = 0

    return words, yaml_keys, False, is_yaml_error

# Aplicar a df_snapshot
metrics = df_snapshot["content"].apply(extract_yaml_and_body_metrics)
df_snapshot["word_count"] = [m[0] for m in metrics]
df_snapshot["frontmatter_keys"] = [m[1] for m in metrics]
df_snapshot["is_empty"] = [m[2] for m in metrics]
df_snapshot["is_yaml_error"] = [m[3] for m in metrics]

# Unir métricas a df_version
df_version_measures = df_version.merge(
    df_snapshot[["source_markdown_file_version_id", "word_count", "frontmatter_keys", "is_empty", "is_yaml_error"]],
    on="source_markdown_file_version_id",
    how="left"
)

In [ ]:
# Unir métricas a la tabla de transiciones
df_trans = df_transitions.merge(
    df_version_measures[["source_markdown_file_version_id", "word_count", "frontmatter_keys"]],
    left_on="source_markdown_file_version_id_curr", right_on="source_markdown_file_version_id"
).merge(
    df_version_measures[["source_markdown_file_version_id", "word_count", "frontmatter_keys"]],
    left_on="source_markdown_file_version_id_prev", right_on="source_markdown_file_version_id",
    suffixes=("_curr", "_prev")
)

# Cálculo de cambios
df_trans["body_length_change"] = df_trans["word_count_curr"] - df_trans["word_count_prev"]
df_trans["abs_body_length_change"] = df_trans["body_length_change"].abs()
df_trans["frontmatter_keys_change"] = df_trans["frontmatter_keys_curr"] - df_trans["frontmatter_keys_prev"]

# Tiempo en días entre versiones
df_trans["time_between_days"] = (df_trans["committed_at_curr"] - df_trans["committed_at_prev"]).dt.total_seconds() / 86400.0

# Documentar transiciones con intervalos negativos (si existieran por fallos de git)
negative_times = (df_trans["time_between_days"] < 0).sum()
print(f"Transiciones con tiempos negativos entre versiones: {negative_times}")

## 4. Agregaciones por Archivo y por Mes

In [ ]:
# 1. Resumen por archivo (historia)
df_first = df_version_measures.sort_values("committed_at").groupby("source_markdown_file_history_id").first().reset_index()
df_last = df_version_measures.sort_values("committed_at").groupby("source_markdown_file_history_id").last().reset_index()

file_summary = df_first[["source_markdown_file_history_id", "committed_at", "word_count"]].rename(
    columns={"committed_at": "first_committed_at", "word_count": "initial_word_count"}
)

file_summary = file_summary.merge(
    df_last[["source_markdown_file_history_id", "committed_at", "word_count"]].rename(
        columns={"committed_at": "last_committed_at", "word_count": "final_word_count"}
    ), on="source_markdown_file_history_id"
)

# Calcular cambio neto y mediana del tiempo
file_summary["net_body_change"] = file_summary["final_word_count"] - file_summary["initial_word_count"]

time_medians = df_trans.groupby("source_markdown_file_history_id_curr")["time_between_days"].median().reset_index()
time_medians.columns = ["source_markdown_file_history_id", "median_time_between_versions_days"]

version_counts = df_version.groupby("source_markdown_file_history_id")["source_markdown_file_version_id"].count().reset_index()
version_counts.columns = ["source_markdown_file_history_id", "version_count"]

file_summary = file_summary.merge(time_medians, on="source_markdown_file_history_id", how="left")
file_summary = file_summary.merge(version_counts, on="source_markdown_file_history_id", how="left")

# 2. Resumen por archivo y mes
df_trans["year_month"] = df_trans["committed_at_curr"].dt.to_period("M").astype(str)

monthly_summary = df_trans.groupby(["source_markdown_file_history_id_curr", "year_month"]).agg(
    transition_count=("source_markdown_file_version_id_curr", "count"),
    median_abs_body_change=("abs_body_length_change", "median")
).reset_index().rename(columns={"source_markdown_file_history_id_curr": "source_markdown_file_history_id"})

# Exportar las 4 tablas a eda/tarea_5/data/processed/
df_version_measures.to_parquet(os.path.join(PROCESSED_DIR, "version_measures.parquet"), index=False)
df_trans.to_parquet(os.path.join(PROCESSED_DIR, "transitions.parquet"), index=False)
file_summary.to_parquet(os.path.join(PROCESSED_DIR, "file_summary.parquet"), index=False)
monthly_summary.to_parquet(os.path.join(PROCESSED_DIR, "monthly_file_summary.parquet"), index=False)

print("¡Todas las tablas agregadas procesadas y guardadas exitosamente!")